# CLIP detector with LayerNorm tuning

Follows GenD ([arXiv 2508.06248](https://arxiv.org/abs/2508.06248)): a CLIP ViT-L/14 image encoder where only the LayerNorm parameters train, L2-normalized 1024-dimensional features, a linear two-class head, and the loss cross entropy + 0.1 alignment + 0.5 uniformity (`face_decision_lab.metric_losses`). Batches keep each real photo together with the fakes made from it (`face_decision_lab.pairs`), and training images are augmented (flip, affine, blur, color jitter, JPEG).

Departures from the paper: `EPOCHS` (paper: two cycles of ten) and the augmentation strengths, which the paper's summary does not give.

Each protocol trains a fresh model, checkpoints every epoch to a private Hub repository and resumes from it, so a lost Colab session costs at most one epoch. A protocol whose report is already on the Hub is skipped.

**Runtime:** Colab Pro, L4 or A100. **Needs** the Colab secret `HF_TOKEN` with write access.

### Configuration

In [ ]:
REPO_SOURCE = "git+https://github.com/paulohenriquevn/face-decision-lab@workspace"
HF_DATASET = "paulohenriquevn/face-decision-data"
HF_MODEL = "paulohenriquevn/face-decision-clip-ln"  # private
CLIP_MODEL = "openai/clip-vit-large-patch14"
PROTOCOL_NAMES = ("unseen_insight", "unseen_text2img")
EPOCHS = 3
BATCH_SIZE = 128
PEAK_LEARNING_RATE = 3e-4
MIN_LEARNING_RATE = 1e-5
AUGMENT = True
# "gend": cross entropy + alignment + uniformity. "ce": cross entropy only, for ablations.
LOSS = "gend"
RUN_TAG = f"{LOSS}-e{EPOCHS}-b{BATCH_SIZE}" + ("" if AUGMENT else "-noaug")  # HF_MODEL/RUN_TAG/<protocol>/
# Test-only datasets for the cross-dataset protocols ("cross_celebdf"), by source_dataset.
CROSS_DATASETS = {"celebdf": "paulohenriquevn/face-decision-celebdf-test"}

### Installation

In [ ]:
%%capture
!pip install -q "face-decision-lab @ {REPO_SOURCE}"

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token = userdata.get("HF_TOKEN"))

### Data

In [ ]:
from face_decision_lab.hub import load_hub_samples

from face_decision_lab.protocol import CROSS_DATASET_PROTOCOLS

samples, hub_images = load_hub_samples(HF_DATASET)
images = {s.image_path: image for s, image in zip(samples, hub_images, strict = True)}
for protocol in PROTOCOL_NAMES:
    if protocol in CROSS_DATASET_PROTOCOLS:
        source = CROSS_DATASET_PROTOCOLS[protocol]
        extra, extra_images = load_hub_samples(CROSS_DATASETS[source], key_prefix = source)
        samples += extra
        images.update({s.image_path: image for s, image in zip(extra, extra_images, strict = True)})
print(len(samples), "crops")

### Model and augmentation

In [ ]:
import io, math, random
import torch
from PIL import Image
from torch import nn
from torchvision import transforms
from transformers import CLIPImageProcessor, CLIPVisionModel

processor = CLIPImageProcessor.from_pretrained(CLIP_MODEL)

def is_layer_norm(name):
    # CLIP vision names: pre_layrnorm, encoder.layers.N.layer_norm1/2, post_layernorm.
    return any(part in name for part in ("layer_norm", "layrnorm", "layernorm"))

class GenDDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = CLIPVisionModel.from_pretrained(CLIP_MODEL)
        # Non-reentrant checkpointing: the reentrant one drops gradients when the input needs none.
        self.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs = {"use_reentrant": False})
        for name, parameter in self.encoder.named_parameters():
            parameter.requires_grad = is_layer_norm(name)
        self.head = nn.Linear(self.encoder.config.hidden_size, 2)

    def forward(self, pixels):
        features = nn.functional.normalize(self.encoder(pixel_values = pixels).pooler_output, dim = -1)
        return self.head(features), features

class RandomJpeg:
    def __init__(self, quality = (60, 95), p = 0.3):
        self.quality, self.p = quality, p

    def __call__(self, image):
        if random.random() >= self.p:
            return image
        buffer = io.BytesIO()
        image.save(buffer, format = "JPEG", quality = random.randint(*self.quality))
        return Image.open(io.BytesIO(buffer.getvalue())).convert("RGB")

augment = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomApply([transforms.RandomAffine(degrees = 5, translate = (0.05, 0.05), scale = (0.95, 1.05))], p = 0.5),
    transforms.RandomApply([transforms.GaussianBlur(5, sigma = (0.1, 1.5))], p = 0.2),
    transforms.RandomApply([transforms.ColorJitter(0.2, 0.2, 0.2, 0.02)], p = 0.5),
    RandomJpeg(),
])

def warmup_cosine(steps_per_epoch, epochs):
    # One epoch of linear warm-up from MIN to PEAK, then cosine decay back to MIN (GenD's cycle shape).
    warmup, total = steps_per_epoch, steps_per_epoch * epochs
    low = MIN_LEARNING_RATE / PEAK_LEARNING_RATE
    def factor(step):
        if step < warmup:
            return low + (1 - low) * step / max(1, warmup)
        progress = (step - warmup) / max(1, total - warmup)
        return low + (1 - low) * 0.5 * (1 + math.cos(math.pi * min(1.0, progress)))
    return factor

probe = GenDDetector()
trainable = sum(p.numel() for p in probe.parameters() if p.requires_grad)
print(f"trainable {trainable:,} of {sum(p.numel() for p in probe.parameters()):,}")
del probe

### Train with per-epoch checkpoints, then score

In [ ]:
import json, time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download
from tqdm.auto import tqdm
from face_decision_lab.metric_losses import gend_loss
from face_decision_lab.pairs import paired_batches
from face_decision_lab.protocol import SEED, evaluation_report, protocol_split
from face_decision_lab.rows import open_image

api = HfApi()
api.create_repo(HF_MODEL, private = True, exist_ok = True)
pool = ThreadPoolExecutor(max_workers = 8)

def pixels(part, train):
    def load(sample):
        image = open_image(images[sample.image_path])
        return augment(image) if train and AUGMENT else image
    return processor(images = list(pool.map(load, part)), return_tensors = "pt")["pixel_values"]

def prefetched(batches, train):
    with ThreadPoolExecutor(max_workers = 1) as loader:
        future = loader.submit(pixels, batches[0], train)
        for index in range(len(batches)):
            current = future.result()
            if index + 1 < len(batches):
                future = loader.submit(pixels, batches[index + 1], train)
            yield batches[index], current

def hub_has(path):
    return path in set(api.list_repo_files(HF_MODEL))

def train_and_score(protocol):
    folder = f"{RUN_TAG}/{protocol}"
    local = Path(f"/content/clip_gend/{protocol}")
    local.mkdir(parents = True, exist_ok = True)
    train, _, test = protocol_split(samples, protocol)
    torch.manual_seed(SEED)
    model = GenDDetector().cuda()
    parameters = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(parameters, lr = PEAK_LEARNING_RATE)
    epochs = [paired_batches(train, BATCH_SIZE, SEED + epoch) for epoch in range(EPOCHS)]
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, warmup_cosine(len(epochs[0]), EPOCHS))
    start = 0
    if hub_has(f"{folder}/checkpoint.pt"):
        state = torch.load(hf_hub_download(HF_MODEL, f"{folder}/checkpoint.pt", force_download = True), weights_only = False)
        model.load_state_dict(state["trainable"], strict = False)
        optimizer.load_state_dict(state["optimizer"]); scheduler.load_state_dict(state["scheduler"])
        start = state["epoch"] + 1
        print(f"{protocol}: resuming after epoch {state['epoch']}")
    started = time.time()
    for epoch in range(start, EPOCHS):
        model.train()
        for step, (batch, x) in enumerate(tqdm(prefetched(epochs[epoch], True), total = len(epochs[epoch]), desc = f"{protocol} epoch {epoch}")):
            y = torch.tensor([0 if s.is_bona_fide else 1 for s in batch], device = "cuda")
            with torch.autocast("cuda", dtype = torch.bfloat16):
                logits, features = model(x.cuda())
            loss = gend_loss(logits.float(), features.float(), y) if LOSS == "gend" else nn.functional.cross_entropy(logits.float(), y)
            loss.backward()
            if step == 0:
                missing = [n for n, p in model.named_parameters() if p.requires_grad and p.grad is None]
                assert not missing, f"no gradient reached {missing[:3]}"
            optimizer.step(); optimizer.zero_grad(); scheduler.step()
            if step % 100 == 0:
                print(f"epoch {epoch} step {step} loss {loss.item():.4f} lr {scheduler.get_last_lr()[0]:.2e}")
        trainable_state = {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}
        torch.save({"trainable": trainable_state, "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(), "epoch": epoch}, local / "checkpoint.pt")
        api.upload_file(path_or_fileobj = str(local / "checkpoint.pt"), path_in_repo = f"{folder}/checkpoint.pt", repo_id = HF_MODEL, commit_message = f"{folder} epoch {epoch}")
    model.eval()
    scores, test_batches = [], [test[i:i + 128] for i in range(0, len(test), 128)]
    with torch.no_grad(), torch.autocast("cuda", dtype = torch.bfloat16):
        for _batch, x in prefetched(test_batches, False):
            logits, _ = model(x.cuda())
            scores += torch.softmax(logits.float(), dim = -1)[:, 1].cpu().tolist()
    report = evaluation_report(test, scores)
    report.update(training_rows = len(train), epochs = EPOCHS, batch_size = BATCH_SIZE, augment = AUGMENT, loss = LOSS, train_minutes_this_session = round((time.time() - started) / 60, 1))
    (local / "report.json").write_text(json.dumps(report, indent = 2))
    api.upload_file(path_or_fileobj = str(local / "report.json"), path_in_repo = f"{folder}/report.json", repo_id = HF_MODEL, commit_message = f"{folder} report")
    return report

reports = {}
for protocol in PROTOCOL_NAMES:
    if hub_has(f"{RUN_TAG}/{protocol}/report.json"):
        print(protocol, "already scored; skipping")
        continue
    reports[protocol] = train_and_score(protocol)
    print(protocol, json.dumps({k: reports[protocol][k] for k in ("auc", "eer", "bpcer_at_max_apcer", "training_rows")}))